# 03 · Conformal upper volatility bounds (design Section 6)

For every panel, horizon, coverage level and base forecaster (DLinear-QLIKE, HAR, GARCH, EWMA): our
asset-standardised bound with ACI (each error is standardised by the median and upper spread Q90 - median of the same
asset's recent closed errors, then one pooled quantile is taken; amendments 1-2), its ablations (Mondrian by class,
mean/sd version), the originally registered vol-of-vol version, and the comparators (pooled split, pooled ACI, per-asset, Mondrian by class, CQR,
parametric). Calibration uses only targets whose window has closed by the forecast date.

The ACI step γ is chosen on the development period (pinball loss) and written to
`results/development/tables/selections.json`; the holdout run reads it and refuses to choose again.

In [ ]:
# ---------------------------------------------------------------- setup (run this cell first)
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/CalVol_project"   # <- change if you placed the folder elsewhere
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_DIR}/requirements.txt"], check=True)
else:
    PROJECT_DIR = os.path.abspath("..")
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))

import json
import numpy as np, pandas as pd
from IPython.display import display, Image, Markdown
from calvol.utils import load_config, get_paths, check_gate
from calvol import pipeline as pp
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 220)

cfg = load_config(PROJECT_DIR)          # mode / sign-off / smoke test are set in config/config.yaml (project: ...)
MODE = check_gate(cfg)                  # refuses a holdout run without sign-off and a frozen commit
P = get_paths(cfg)
PER = pp.eval_split(cfg)
print(f"MODE = {MODE.upper()}  |  evaluation period = {PER}  |  smoke test = {cfg['project']['smoke_test']}")
print("Results folder:", P.results)

def show(name):
    f = P.figures / f"{name}.png"
    if f.exists():
        display(Image(filename=str(f), width=760))
    else:
        print("(figure not produced:", name, ")")

def table(name, round_=4):
    f = P.tables / f"{name}.csv"
    if f.exists():
        display(pd.read_csv(f).round(round_))
    else:
        print("(table not produced:", name, ")")

In [ ]:
JOBS = [("assets", h) for h in cfg["panels"]["assets"]["horizons"]] + [(p, 21) for p in pp.FACTOR_PANELS]
for name, h in JOBS:
    b = pp.stage_bounds(cfg, name, h)
    print(name, h, b.shape)
sel = pp.load_selections(cfg)
print(json.dumps(sel, indent=1))

## γ selection on the development period

In [ ]:
for name, h in JOBS:
    f = P.dev_results / "tables" / f"{name}_h{h}_gamma_selection.csv"
    if f.exists():
        display(Markdown(f"**{name}, h = {h}**")); display(pd.read_csv(f).round(5))